# 1: Imports and Path Setup

In [1]:
import os
import sys
import numpy as np
import tensorflow as tf

In [2]:
PROJECT_ROOT = '/content/drive/MyDrive/Transformer_Mine'

In [6]:
from google.colab import drive
import os

# Ensure the mount point is completely clean before attempting to mount.
# This forcefully removes any existing /content/drive directory.
if os.path.exists('/content/drive'):
    !rm -rf /content/drive

# Create an empty directory for the mount point
os.makedirs('/content/drive', exist_ok=True)

drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Transformer_Mine

Mounted at /content/drive
/content/drive/MyDrive/Transformer_Mine


In [7]:
tf.__version__

'2.20.0'

In [8]:
PROJECT_ROOT

'/content/drive/MyDrive/Transformer_Mine'

# 2: Mathematical Foundation

From the paper (Section 3.3):

$$
FFN(x) = \max(0, xW_1 + b_1)W_2 + b_2
$$

Where:

- `x`   : input,  shape `(batch, seq_len, d_model)`
- `W1`  : weight, shape `(d_model, d_ff)`
- `b1`  : bias,   shape `(d_ff,)`
- `W2`  : weight, shape `(d_ff, d_model)`
- `b2`  : bias,   shape `(d_model,)`

The inner dimension `d_ff` is larger than `d_model`.

This expansion and contraction gives the FFN capacity to  
learn complex non-linear transformations per position.

### Key points:

- Applied **INDEPENDENTLY** to each position (position-wise)
- Same weights across all positions (not position-specific)
- Two linear transformations with one ReLU in between
- No attention — positions do not interact here

### Transformer Base:

- `d_model = 512`
- `d_ff    = 2048` (4x expansion)

### Parameter count:

- `W1` : `512 × 2048 = 1,048,576`
- `b1` : `2048 = 2,048`
- `W2` : `2048 × 512 = 1,048,576`
- `b2` : `512 = 512`
- **Total** : `2,099,712` per FFN layer

In [9]:
d_model = 512
d_ff = 2048

w1_params = d_model * d_ff
w2_params = d_ff * d_model
b1_params = d_ff
b2_params = d_model
total = w1_params + w2_params + b1_params + b2_params

In [10]:
print(f"  W1 : {d_model} x {d_ff} = {w1_params:,}")
print(f"  b1 : {d_ff} = {b1_params:,}")
print(f"  W2 : {d_ff} x {d_model} = {w2_params:,}")
print(f"  b2 : {d_model} = {b2_params:,}")
print(f"  Total per FFN layer = {total:,}")

  W1 : 512 x 2048 = 1,048,576
  b1 : 2048 = 2,048
  W2 : 2048 x 512 = 1,048,576
  b2 : 512 = 512
  Total per FFN layer = 2,099,712


# CELL 3: FeedForwardNetwork Implementation

Implemented as a `tf.keras.layers.Layer`.

Uses two Dense layers with ReLU activation between them.

Dropout is applied after the first activation.

In [12]:
class FeedForwardNetwork(tf.keras.layers.Layer):
    """
    Position-wise feed-forward network.
    FFN(x) = max(0, x W1 + b1) W2 + b2

    Applied independently to each position in the sequence.
    The same weights are shared across all positions.

    Args:
        d_model : input and output dimension (e.g. 512)
        d_ff    : inner dimension (e.g. 2048)
        dropout : dropout rate (paper: 0.1)
    """
    def __init__(self, d_model, d_ff, dropout = 0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.d_ff = d_ff
        self.dropout = dropout
        self.supports_masking = True

        # First linear transformation + ReLU
        self.dense_1 = tf.keras.layers.Dense(
            d_ff,
            activation = 'relu',
            name = 'ffn_dense_1'
        )
        # input  shape: (batch, seq_len, d_model)
        # output shape: (batch, seq_len, d_ff)

        # Second linear transformation
        self.dense_2 = tf.keras.layers.Dense(
            d_model,
            name='ffn_dense_2',
        )
        # input  shape: (batch, seq_len, d_ff)
        # output shape: (batch, seq_len, d_model)

        self.dropout = tf.keras.layers.Dropout(dropout)

    def call(self, x, training = False):
        """
        Forward pass.

        Args:
            x        : (batch, seq_len, d_model)
            training : controls dropout

        Returns:
            (batch, seq_len, d_model)
        """

        # x shape: (batch, seq_len, d_model)
        x = self.dense_1(x)
        # x shape: (batch, seq_len, d_ff)

        x = self.dropout(x, training = training)

        x = self.dense_2(x)
        # x shape: (batch, seq_len, d_model)

        return x

    def get_config(self):
        config = super().get_config()

        config.update({
            'd_model' : self.d_model,
            'd_ff' : self.d_ff,
            'dropout' : self.dropout_rate,
        })

        return config



#CELL 4: Shape Tests

In [13]:
D_MODEL = 512
D_FF    = 2048
BATCH   = 2
SEQ_LEN = 10

In [14]:
tf.random.set_seed(42)

In [15]:
ffn = FeedForwardNetwork(d_model=D_MODEL, d_ff=D_FF, dropout=0.0)

In [16]:
inputs = tf.keras.Input(shape=(SEQ_LEN, D_MODEL))
outputs = ffn(inputs)
model_for_summary = tf.keras.Model(inputs=inputs, outputs=outputs)
model_for_summary.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feed_forward_network            │ (None, 10, 512)        │     2,099,712 │
│ (FeedForwardNetwork)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,099,712 (8.01 MB)

 Trainable params: 2,099,712 (8.01 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
x = tf.random.normal((BATCH, SEQ_LEN, D_MODEL))

In [18]:
output = ffn(x, training=False)

In [19]:
assert output.shape == (BATCH, SEQ_LEN, D_MODEL), \
    f"Output shape wrong: {output.shape}"

print(f"input shape : {x.shape}")
print(f"output shape : {output.shape}")
print("output shape matches input shape (batch, seq_len, d_model) - correct")


input shape : (2, 10, 512)
output shape : (2, 10, 512)
output shape matches input shape (batch, seq_len, d_model) - correct


In [20]:
for seq in [1, 5, 50, 128]:

    x_var = tf.random.normal((BATCH, seq, D_MODEL))
    out = ffn(x_var, training=False)

    assert out.shape == (BATCH, seq, D_MODEL), \
        f"Shape wrong for seq_len={seq}: {out.shape}"

    print(f"seq_len={seq:<4} output shape: {out.shape} - correct")


seq_len=1    output shape: (2, 1, 512) - correct
seq_len=5    output shape: (2, 5, 512) - correct
seq_len=50   output shape: (2, 50, 512) - correct
seq_len=128  output shape: (2, 128, 512) - correct


#CELL 5: Parameter Count Verification

In [21]:
# Force weight creation
_ = ffn(tf.zeros((1, 1, D_MODEL)))

In [22]:
total_params = 0
for w in ffn.trainable_weights:
    n = tf.size(w).numpy()
    total_params += n
    print(f"{w.name:<30} shape={str(w.shape):<15} params={n:,}")

expected = D_MODEL * D_FF + D_FF + D_FF * D_MODEL + D_MODEL
print(f"\nTotal trainable params : {total_params:,}")
print(f"Expected : {expected:,}")

assert total_params == expected, \
    f"Parameter count mismatch: {total_params} vs {expected}"

print("Parameter count correct.")

kernel                         shape=(512, 2048)     params=1,048,576
bias                           shape=(2048,)         params=2,048
kernel                         shape=(2048, 512)     params=1,048,576
bias                           shape=(512,)          params=512

Total trainable params : 2,099,712
Expected : 2,099,712
Parameter count correct.


# CELL 6: Numerical Behavior Tests

Verifies:

1. ReLU removes negative activations after first dense layer
2. Output is position-wise (each position processed independently)
3. Dropout has no effect in inference mode (`training=False`)
4. Output changes with `training=True` (dropout active)

In [23]:
tf.random.set_seed(0)

In [24]:
ffn_test = FeedForwardNetwork(d_model=D_MODEL, d_ff=D_FF, dropout=0.5)

In [25]:
x_test = tf.random.normal((2, 8, D_MODEL))

Test 1: ReLU - intermediate activations should have no negatives

In [26]:
with tf.GradientTape() as tape:
    intermediate = ffn_test.dense_1(x_test)

min_intermediate = tf.reduce_min(intermediate).numpy()

print(f"min value after dense_1+relu: {min_intermediate:.6f}")

assert min_intermediate >= 0.0, \
    f"ReLU should remove negatives, got min={min_intermediate}"
print("no negative activations after ReLU - correct")

min value after dense_1+relu: 0.000000
no negative activations after ReLU - correct


Test 2: Position-wise independence

In [27]:
x_modified = x_test.numpy().copy()
x_modified[0, 3] = 999.0   # drastically change position 3
x_modified = tf.constant(x_modified)

In [28]:
out_original = ffn_test(x_test, training=False).numpy()
out_modified = ffn_test(x_modified, training=False).numpy()

In [29]:
diff_pos0 = abs(out_original[0, 0] - out_modified[0, 0]).max()
diff_pos3 = abs(out_original[0, 3] - out_modified[0, 3]).max()

In [30]:
print(f"  max diff at position 0 (unchanged): {diff_pos0:.6f}")
print(f"  max diff at position 3 (changed)  : {diff_pos3:.6f}")
assert diff_pos0 < 1e-5, "Position 0 should be unaffected"
assert diff_pos3 > 1e-3, "Position 3 should differ significantly"

print("positions are processed independently - correct")

  max diff at position 0 (unchanged): 0.000000
  max diff at position 3 (changed)  : 2486.343262
positions are processed independently - correct


Test 3: No dropout at inference

In [31]:
out_inf_1 = ffn_test(x_test, training=False).numpy()
out_inf_2 = ffn_test(x_test, training=False).numpy()

In [32]:
max_diff_inf = abs(out_inf_1 - out_inf_2).max()

In [33]:
print(f"max diff between two inference passes: {max_diff_inf:.2e}")

assert max_diff_inf < 1e-6, "Inference should be deterministic"

print("deterministic at inference - correct")


max diff between two inference passes: 0.00e+00
deterministic at inference - correct


Test 4: Dropout active during training

In [34]:
out_train_1 = ffn_test(x_test, training=True).numpy()
out_train_2 = ffn_test(x_test, training=True).numpy()

In [35]:
max_diff_train = abs(out_train_1 - out_train_2).max()

In [36]:
print(f"max diff between two training passes: {max_diff_train:.6f}")

assert max_diff_train > 0.0, "Training passes should differ due to dropout"

print("stochastic during training - correct")


max diff between two training passes: 3.903629
stochastic during training - correct


#CELL 7: Gradient Flow Test

In [37]:
tf.random.set_seed(42)

In [38]:
ffn_grad = FeedForwardNetwork(d_model=512, d_ff=2048, dropout=0.0)

In [39]:
x_grad   = tf.random.normal((2, 6, 512))

In [40]:

with tf.GradientTape() as tape:
    tape.watch(x_grad)
    output_g = ffn_grad(x_grad, training=False)
    loss = tf.reduce_sum(output_g)

input_grad = tape.gradient(loss, x_grad)
weight_grads = []

with tf.GradientTape() as tape2:
    out2 = ffn_grad(x_grad, training=False)
    loss2 = tf.reduce_sum(out2)

weight_grads = tape2.gradient(loss2, ffn_grad.trainable_variables)

print("\nGradients w.r.t. input:")
if input_grad is None:
    print("gradient is None - backprop broken")
else:
    norm = tf.norm(input_grad).numpy()
    print(f"  shape: {input_grad.shape}  norm: {norm:.4f}")
    assert norm > 0

print("\nGradients w.r.t. weights:")

all_ok = True
for w, g in zip(ffn_grad.trainable_variables, weight_grads):
    if g is None:
        print(f"{w.name}: None - broken")
        all_ok = False
    else:
        norm = tf.norm(g).numpy()
        has_nan = tf.reduce_any(tf.math.is_nan(g)).numpy()
        status  = "ok" if (norm > 0 and not has_nan) else "FAILED"
        print(f"{w.name:<35} norm={norm:.4f}  {status}")
        if norm == 0 or has_nan:
            all_ok = False

if all_ok:
    print("\nGradient flow test passed.")
else:
    print("\nGradient flow test FAILED.")


Gradients w.r.t. input:
  shape: (2, 6, 512)  norm: 44.5808

Gradients w.r.t. weights:
kernel                              norm=1530.7770  ok
bias                                norm=177.6300  ok
kernel                              norm=3325.5510  ok
bias                                norm=271.5290  ok

Gradient flow test passed.


# CELL 8: Comparison with Paper Specification

Explicitly documents how our implementation maps to the  
paper and flags any deviations.

In [41]:
rows = [
    ("Formula", "FFN(x) = max(0, xW1+b1)W2 + b2", "identical"),
    ("d_model","512","512"),
    ("d_ff", "2048","2048"),
    ("Activation", "ReLU","ReLU (Dense activation)"),
    ("Dropout","applied (rate=0.1)", "after first dense layer"),
    ("Position-wise", "yes — same W per position", "yes — Dense applies to last dim"),
    ("Parameters", "2,099,712 per layer", f"{expected:,} per layer"),
    ("Bias", "yes", "yes (Keras Dense default)"),
]

print(f"\n{'Component':<20} {'Paper':<35} {'Ours':<35}")
for component, paper, ours in rows:
    print(f"{component:<20} {paper:<35} {ours:<35}")



print("\nDeviations: none.")





Component            Paper                               Ours                               
Formula              FFN(x) = max(0, xW1+b1)W2 + b2      identical                          
d_model              512                                 512                                
d_ff                 2048                                2048                               
Activation           ReLU                                ReLU (Dense activation)            
Dropout              applied (rate=0.1)                  after first dense layer            
Position-wise        yes — same W per position           yes — Dense applies to last dim    
Parameters           2,099,712 per layer                 2,099,712 per layer                
Bias                 yes                                 yes (Keras Dense default)          

Deviations: none.


#CELL 9: Save FeedForwardNetwork to Drive

In [42]:
FFN_PY = os.path.join(PROJECT_ROOT, 'feed_forward.py')

In [43]:
code = '''"""
feed_forward.py - Position-wise Feed-Forward Network
Transformer Reproduction Project - Phase 7

Usage:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Transformer_Mine
    from feed_forward import FeedForwardNetwork
"""

import tensorflow as tf

class FeedForwardNetwork(tf.keras.layers.Layer):
    """
    Position-wise feed-forward network.

    FFN(x) = max(0, x W1 + b1) W2 + b2

    Args:
        d_model : input and output dimension (e.g. 512)
        d_ff : inner dimension (e.g. 2048)
        dropout : dropout rate (paper: 0.1)
    """

    def __init__(self, d_model, d_ff, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.d_ff = d_ff
        self.dropout_rate = dropout
        self.supports_masking = True
        self.dense_1 = tf.keras.layers.Dense(d_ff, activation="relu", name="ffn_dense_1")
        self.dense_2 = tf.keras.layers.Dense(d_model, name="ffn_dense_2")
        self.dropout = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=False):
        # x shape: (batch, seq_len, d_model)
        x = self.dense_1(x)
        # x shape: (batch, seq_len, d_ff)
        x = self.dropout(x, training=training)
        x = self.dense_2(x)
        # x shape: (batch, seq_len, d_model)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "d_model" : self.d_model,
            "d_ff" : self.d_ff,
            "dropout" : self.dropout_rate,
        })
        return config
'''


In [44]:
import os
os.makedirs(os.path.dirname(FFN_PY), exist_ok=True)
with open(FFN_PY, 'w', encoding='utf-8') as f:
    f.write(code)

In [45]:
import importlib
import feed_forward
importlib.reload(feed_forward)
from feed_forward import FeedForwardNetwork as FFN_loaded

ffn_reload = FFN_loaded(d_model=512, d_ff=2048, dropout=0.0)
x_r = tf.random.normal((2, 10, 512))
out_r = ffn_reload(x_r, training=False)

assert out_r.shape == (2, 10, 512)

total_r = sum(tf.size(w).numpy() for w in ffn_reload.trainable_variables)
assert total_r == 2 * 512 * 2048 + 2048 + 512

print("feed_forward.py saved and reloaded.")
print(f"path : {FFN_PY}")
print(f"output shape : {out_r.shape}")
print(f"total params : {total_r:,}")

feed_forward.py saved and reloaded.
path : /content/drive/MyDrive/Transformer_Mine/feed_forward.py
output shape : (2, 10, 512)
total params : 2,099,712
